# Klasifikasi Tingkat Stres Mahasiswa Menggunakan SVM

Notebook ini disesuaikan untuk dataset baru `StressLevelDataset.csv`. Alur kerja mencakup inspeksi awal, preprocessing, handling outlier, EDA, korelasi Spearman, training Support Vector Machine (SVM), dan evaluasi model klasifikasi `stress_level`.

In [ ]:
# Import library utama untuk analisis data, visualisasi, preprocessing, modeling, dan evaluasi.
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Pengaturan agar output notebook lebih rapi dan visual konsisten.
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 1. Load Dataset dan Inspeksi Awal

In [ ]:
# Load dataset baru dari folder Dataset.
DATA_PATH = Path("Dataset") / "StressLevelDataset.csv"
TARGET_COL = "stress_level"

stress_labels = {
    0: "Low Stress",
    1: "Medium Stress",
    2: "High Stress",
}

df = pd.read_csv(DATA_PATH)

print(f"Dimensi dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
display(df.head())

In [ ]:
# Menampilkan informasi struktur dataset: nama kolom, jumlah non-null, dan tipe data.
df.info()

In [ ]:
# Mengecek jumlah missing values pada setiap kolom.
missing_values = df.isna().sum().to_frame(name="missing_count")
display(missing_values)

## 2. Handling Missing Values

In [ ]:
# Dataset baru saat dicek tidak memiliki missing values.
# Cell ini tetap dibuat agar pipeline preprocessing lengkap dan aman jika data diperbarui.
df_clean = df.copy()

# Jika target kosong, baris tersebut tidak bisa digunakan untuk supervised learning sehingga dihapus.
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=[TARGET_COL]).copy()
rows_after = len(df_clean)

# Semua fitur pada dataset ini numerik. Jika ada missing value di fitur, imputasi dengan median.
feature_cols = [col for col in df_clean.columns if col != TARGET_COL]
numeric_feature_cols = df_clean[feature_cols].select_dtypes(include=np.number).columns.tolist()

for col in numeric_feature_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)

# Jika pada versi data lain ada kolom kategorikal, imputasi dengan modus.
categorical_feature_cols = df_clean[feature_cols].select_dtypes(exclude=np.number).columns.tolist()
for col in categorical_feature_cols:
    mode_value = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_value)

print(f"Baris sebelum drop target kosong: {rows_before}")
print(f"Baris setelah drop target kosong : {rows_after}")
print(f"Jumlah fitur numerik yang diimputasi median    : {len(numeric_feature_cols)}")
print(f"Jumlah fitur kategorikal yang diimputasi modus : {len(categorical_feature_cols)}")

display(df_clean.isna().sum().to_frame(name="missing_after_imputation"))

## 3. Handling Outliers Menggunakan IQR Clipping

In [ ]:
# Kolom skor numerik yang paling relevan untuk dicek outlier-nya.
# Skor ordinal 0-5 seperti study_load atau bullying tidak di-clipping agar nilai kategorinya tetap utuh.
outlier_cols = ["anxiety_level", "self_esteem", "depression"]

# Visualisasi boxplot sebelum clipping outlier.
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(18, 4))

for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df_clean[col], ax=ax, color="#76b7b2")
    ax.set_title(f"{col} - Sebelum Clipping")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Fungsi clipping outlier dengan metode IQR.
# Baris data tidak dihapus; nilai ekstrem hanya dibatasi ke lower_bound atau upper_bound.
def clip_outliers_iqr(dataframe, columns):
    df_clipped = dataframe.copy()
    bounds = {}

    for col in columns:
        q1 = df_clipped[col].quantile(0.25)
        q3 = df_clipped[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        bounds[col] = {
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Min_Before": df_clipped[col].min(),
            "Max_Before": df_clipped[col].max(),
        }

        df_clipped[col] = df_clipped[col].clip(lower=lower_bound, upper=upper_bound)
        bounds[col]["Min_After"] = df_clipped[col].min()
        bounds[col]["Max_After"] = df_clipped[col].max()

    bounds_df = pd.DataFrame(bounds).T
    return df_clipped, bounds_df


df_processed, iqr_bounds = clip_outliers_iqr(df_clean, outlier_cols)

print(f"Jumlah baris sebelum clipping: {len(df_clean)}")
print(f"Jumlah baris setelah clipping: {len(df_processed)}")
display(iqr_bounds)

In [ ]:
# Visualisasi boxplot setelah clipping untuk melihat dampak pembatasan nilai outlier.
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(18, 4))

for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df_processed[col], ax=ax, color="#59a14f")
    ax.set_title(f"{col} - Setelah Clipping")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Visualisasi 1: Distribusi target stress_level.
plt.figure(figsize=(8, 5))
stress_order = sorted(df_processed[TARGET_COL].unique())
sns.countplot(data=df_processed, x=TARGET_COL, order=stress_order, palette="viridis")
plt.title("Distribusi stress_level")
plt.xlabel("stress_level")
plt.ylabel("Jumlah Mahasiswa")
plt.xticks(ticks=range(len(stress_order)), labels=[f"{i} - {stress_labels.get(i, i)}" for i in stress_order])
plt.tight_layout()
plt.show()

display(df_processed[TARGET_COL].value_counts().sort_index().rename(index=stress_labels).to_frame(name="count"))

In [ ]:
# Visualisasi 2: Hubungan kualitas tidur dengan tingkat stres.
# Nilai sleep_quality yang lebih tinggi merepresentasikan kualitas tidur yang lebih baik.
sleep_summary = (
    df_processed.groupby("sleep_quality")[TARGET_COL]
    .agg(["count", "mean", "median"])
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=sleep_summary, x="sleep_quality", y="mean", ax=axes[0], color="#4e79a7")
axes[0].set_title("Rata-rata stress_level per sleep_quality")
axes[0].set_xlabel("sleep_quality")
axes[0].set_ylabel("Rata-rata stress_level")

sns.countplot(data=df_processed, x="sleep_quality", hue=TARGET_COL, ax=axes[1], palette="viridis")
axes[1].set_title("Distribusi stress_level berdasarkan sleep_quality")
axes[1].set_xlabel("sleep_quality")
axes[1].set_ylabel("Jumlah Mahasiswa")
axes[1].legend(title="stress_level")

plt.tight_layout()
plt.show()

display(sleep_summary)

In [ ]:
# Visualisasi 3: KDE plot distribusi depression berdasarkan stress_level.
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df_processed,
    x="depression",
    hue=TARGET_COL,
    common_norm=False,
    palette="viridis",
    linewidth=2,
    fill=False,
)
plt.title("Distribusi Kepadatan Depression Berdasarkan stress_level")
plt.xlabel("depression")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi tambahan: hubungan anxiety_level dan depression terhadap stress_level.
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_processed,
    x="anxiety_level",
    y="depression",
    hue=TARGET_COL,
    palette="viridis",
    alpha=0.75,
)
plt.title("Anxiety Level vs Depression Berdasarkan stress_level")
plt.xlabel("anxiety_level")
plt.ylabel("depression")
plt.legend(title="stress_level")
plt.tight_layout()
plt.show()

## 5. Data Encoding

In [ ]:
# Dataset baru sudah menggunakan representasi numerik untuk semua fitur.
# Cell ini tetap menangani kemungkinan kolom kategorikal jika dataset diperbarui di kemudian hari.
df_encoded = df_processed.copy()

categorical_cols = df_encoded.drop(columns=[TARGET_COL]).select_dtypes(exclude=np.number).columns.tolist()

if categorical_cols:
    print("Kolom kategorikal terdeteksi dan akan di-one-hot encode:", categorical_cols)
    df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True, dtype=int)
else:
    print("Tidak ada kolom kategorikal. One-Hot Encoding tidak diperlukan.")

# Memastikan target tetap bertipe integer.
df_encoded[TARGET_COL] = df_encoded[TARGET_COL].astype(int)

display(df_encoded.head())
display(df_encoded.isna().sum()[df_encoded.isna().sum() > 0].to_frame(name="missing_after_encoding"))

## 6. Correlation Matrix Spearman

In [ ]:
# Menghitung korelasi Spearman karena banyak fitur berupa skor ordinal.
corr_matrix = df_encoded.corr(method="spearman")

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, linewidths=0.3, linecolor="white")
plt.title("Heatmap Korelasi Spearman")
plt.tight_layout()
plt.show()

# Menampilkan fitur dengan korelasi absolut paling kuat terhadap stress_level.
stress_corr = corr_matrix[TARGET_COL].drop(labels=TARGET_COL)
strongest_corr = stress_corr.reindex(stress_corr.abs().sort_values(ascending=False).index)

print("10 fitur dengan korelasi Spearman absolut terkuat terhadap stress_level:")
display(strongest_corr.head(10).to_frame(name="spearman_correlation"))

## 7. Data Splitting dan Standard Scaling

In [ ]:
# Memisahkan fitur (X) dan target (y).
X = df_encoded.drop(columns=TARGET_COL)
y = df_encoded[TARGET_COL]

# Baseline sederhana: akurasi jika selalu menebak kelas mayoritas.
majority_baseline = y.value_counts(normalize=True).max()
print(f"Baseline kelas mayoritas: {majority_baseline:.4f}")

# Split data 80:20 dengan stratify agar proporsi kelas target tetap terjaga.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

# StandardScaler wajib untuk SVM agar semua fitur berada pada skala yang sebanding.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Ukuran X_train: {X_train.shape}")
print(f"Ukuran X_test : {X_test.shape}")
print(f"Ukuran y_train: {y_train.shape}")
print(f"Ukuran y_test : {y_test.shape}")

print("\nDistribusi kelas pada y_train:")
display(y_train.value_counts(normalize=True).sort_index().rename(index=stress_labels).to_frame(name="train_proportion"))

print("Distribusi kelas pada y_test:")
display(y_test.value_counts(normalize=True).sort_index().rename(index=stress_labels).to_frame(name="test_proportion"))

## 8. Model Training Menggunakan SVM

In [ ]:
# Melatih model SVM dengan kernel RBF dan decision function One-vs-Rest untuk multiclass.
svm_model = SVC(
    kernel="rbf",
    decision_function_shape="ovr",
    random_state=RANDOM_STATE,
)

svm_model.fit(X_train_scaled, y_train)

print("Model SVM selesai dilatih.")

## 9. Evaluasi Model

In [ ]:
# Melakukan prediksi pada data testing.
y_pred = svm_model.predict(X_test_scaled)

# Menghitung dan menampilkan akurasi keseluruhan.
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi keseluruhan model SVM: {accuracy:.4f}")
print(f"Baseline kelas mayoritas        : {majority_baseline:.4f}")
print(f"Selisih terhadap baseline       : {accuracy - majority_baseline:.4f}")

# Menampilkan classification report: precision, recall, dan F1-score per kelas.
target_names = [stress_labels[i] for i in sorted(y.unique())]
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names, digits=4, zero_division=0))

In [ ]:
# Visualisasi confusion matrix menggunakan heatmap.
class_labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=class_labels)
cm_df = pd.DataFrame(
    cm,
    index=[stress_labels[i] for i in class_labels],
    columns=[stress_labels[i] for i in class_labels],
)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix - SVM")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 10. Validasi Tambahan dengan Cross-Validation

In [ ]:
# Cross-validation membantu melihat apakah performa model stabil di beberapa split data.
# Scaling dilakukan di dalam tiap fold untuk mencegah data leakage.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_fold_train = X.iloc[train_idx]
    X_fold_valid = X.iloc[valid_idx]
    y_fold_train = y.iloc[train_idx]
    y_fold_valid = y.iloc[valid_idx]

    fold_scaler = StandardScaler()
    X_fold_train_scaled = fold_scaler.fit_transform(X_fold_train)
    X_fold_valid_scaled = fold_scaler.transform(X_fold_valid)

    fold_model = SVC(kernel="rbf", decision_function_shape="ovr", random_state=RANDOM_STATE)
    fold_model.fit(X_fold_train_scaled, y_fold_train)
    fold_pred = fold_model.predict(X_fold_valid_scaled)

    fold_accuracy = accuracy_score(y_fold_valid, fold_pred)
    cv_scores.append(fold_accuracy)
    print(f"Fold {fold}: {fold_accuracy:.4f}")

print(f"\nRata-rata akurasi CV: {np.mean(cv_scores):.4f}")
print(f"Standar deviasi CV  : {np.std(cv_scores):.4f}")

## 11. Pengecekan Kualitas Data Tambahan

Dataset baru sudah bersih, tetapi sebelum membandingkan banyak algoritma perlu ditambahkan beberapa validasi: duplikat, keseimbangan kelas, jumlah nilai unik per fitur, dan baseline mayoritas. Untuk dataset ini, SMOTE atau resampling tidak diperlukan karena distribusi target sudah relatif seimbang.

In [ ]:
# Pengecekan tambahan untuk memastikan kualitas data sebelum model comparison.
duplicate_count = df_encoded.duplicated().sum()
total_missing = df_encoded.isna().sum().sum()
target_distribution = df_encoded[TARGET_COL].value_counts().sort_index()
target_distribution_pct = df_encoded[TARGET_COL].value_counts(normalize=True).sort_index() * 100

quality_summary = pd.DataFrame({
    "metric": [
        "Jumlah baris",
        "Jumlah kolom",
        "Total missing values",
        "Jumlah duplikat",
        "Jumlah kelas target",
        "Baseline kelas mayoritas",
    ],
    "value": [
        df_encoded.shape[0],
        df_encoded.shape[1],
        total_missing,
        duplicate_count,
        df_encoded[TARGET_COL].nunique(),
        f"{majority_baseline:.4f}",
    ],
})

display(quality_summary)

print("Distribusi target:")
display(
    pd.DataFrame({
        "label": target_distribution.index.map(stress_labels),
        "count": target_distribution.values,
        "percentage": target_distribution_pct.round(2).values,
    })
)

print("Jumlah nilai unik setiap kolom:")
display(df_encoded.nunique().sort_values().to_frame(name="n_unique"))

## 12. Perbandingan Banyak Algoritma

Model dibandingkan menggunakan `StratifiedKFold` pada data train saja. Pemilihan model terbaik memakai `f1_macro`, karena target memiliki 3 kelas dan kita ingin performa seimbang di semua kelas. Test set tidak digunakan untuk memilih model, hanya untuk evaluasi akhir.

In [ ]:
# Import algoritma tambahan dan utilitas evaluasi untuk model comparison.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import precision_score, recall_score, f1_score, make_scorer


def scaled_pipeline(model):
    # Pipeline untuk model yang sensitif terhadap skala fitur.
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


def tree_pipeline(model):
    # Pipeline untuk model tree-based yang tidak membutuhkan scaling.
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])


model_pipelines = {
    "Dummy Majority": tree_pipeline(DummyClassifier(strategy="most_frequent")),
    "Logistic Regression": scaled_pipeline(LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    "KNN": scaled_pipeline(KNeighborsClassifier(n_neighbors=7)),
    "Gaussian NB": scaled_pipeline(GaussianNB()),
    "Decision Tree": tree_pipeline(DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)),
    "Random Forest": tree_pipeline(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Extra Trees": tree_pipeline(ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting": tree_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE)),
    "Hist Gradient Boosting": tree_pipeline(HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    "AdaBoost": tree_pipeline(AdaBoostClassifier(random_state=RANDOM_STATE)),
    "SVM Linear": scaled_pipeline(SVC(kernel="linear", decision_function_shape="ovr", random_state=RANDOM_STATE)),
    "SVM RBF": scaled_pipeline(SVC(kernel="rbf", decision_function_shape="ovr", random_state=RANDOM_STATE)),
    "MLP Neural Network": scaled_pipeline(
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=1000,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )
    ),
}

# Scorer dibuat eksplisit agar precision tidak bermasalah saat baseline dummy tidak memprediksi semua kelas.
scoring = {
    "accuracy": "accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
}

cv_compare = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
comparison_rows = []

for model_name, pipeline in model_pipelines.items():
    cv_result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_compare,
        scoring=scoring,
        return_train_score=False,
        n_jobs=None,
        error_score="raise",
    )

    comparison_rows.append({
        "Model": model_name,
        "Accuracy Mean": cv_result["test_accuracy"].mean(),
        "Precision Macro Mean": cv_result["test_precision_macro"].mean(),
        "Recall Macro Mean": cv_result["test_recall_macro"].mean(),
        "F1 Macro Mean": cv_result["test_f1_macro"].mean(),
        "F1 Macro Std": cv_result["test_f1_macro"].std(),
    })

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(by=["F1 Macro Mean", "Accuracy Mean"], ascending=False)
    .reset_index(drop=True)
)

best_model_name = comparison_df.loc[0, "Model"]

print(f"Model terbaik berdasarkan rata-rata F1 Macro CV: {best_model_name}")
display(comparison_df.style.format({
    "Accuracy Mean": "{:.4f}",
    "Precision Macro Mean": "{:.4f}",
    "Recall Macro Mean": "{:.4f}",
    "F1 Macro Mean": "{:.4f}",
    "F1 Macro Std": "{:.4f}",
}))

## 13. Training dan Evaluasi Model Terbaik

Model terbaik dari cross-validation dilatih ulang pada seluruh data train, lalu dievaluasi satu kali pada test set.

In [ ]:
# Melatih ulang model terbaik menggunakan seluruh data train.
best_model = model_pipelines[best_model_name]
best_model.fit(X_train, y_train)

# Evaluasi akhir pada test set.
best_y_pred = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, best_y_pred)
best_f1_macro = f1_score(y_test, best_y_pred, average="macro")

print(f"Model terbaik              : {best_model_name}")
print(f"Test Accuracy             : {best_accuracy:.4f}")
print(f"Test F1 Macro             : {best_f1_macro:.4f}")
print(f"Baseline kelas mayoritas  : {majority_baseline:.4f}")
print(f"Kenaikan akurasi baseline : {best_accuracy - majority_baseline:.4f}")

print("\nClassification Report Model Terbaik:")
print(classification_report(y_test, best_y_pred, target_names=target_names, digits=4, zero_division=0))

In [ ]:
# Confusion matrix untuk model terbaik.
best_cm = confusion_matrix(y_test, best_y_pred, labels=class_labels)
best_cm_df = pd.DataFrame(
    best_cm,
    index=[stress_labels[i] for i in class_labels],
    columns=[stress_labels[i] for i in class_labels],
)

plt.figure(figsize=(8, 6))
sns.heatmap(best_cm_df, annot=True, fmt="d", cmap="Greens", cbar=False)
plt.title(f"Confusion Matrix - Model Terbaik ({best_model_name})")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 14. Interpretasi Model dengan Permutation Importance

Permutation importance digunakan agar interpretasi fitur tetap bisa dilakukan untuk algoritma apa pun yang terpilih sebagai model terbaik.

In [ ]:
# Mengukur kontribusi fitur terhadap performa model terbaik dengan permutation importance.
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": perm_importance.importances_mean,
        "importance_std": perm_importance.importances_std,
    })
    .sort_values(by="importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("15 fitur paling penting berdasarkan permutation importance:")
display(importance_df.head(15))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=importance_df.head(15),
    x="importance_mean",
    y="feature",
    color="#4e79a7",
)
plt.title(f"Top 15 Feature Importance - {best_model_name}")
plt.xlabel("Mean decrease in F1 Macro")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Catatan Preprocessing

Preprocessing tambahan yang relevan untuk dataset ini adalah validasi duplikat, validasi class balance, penggunaan `Pipeline` agar imputer/scaler hanya fit pada data train, dan pemilihan model berbasis cross-validation. Karena dataset tidak memiliki missing value, tidak memiliki duplikat, dan kelas target seimbang, tidak perlu menambahkan SMOTE. Karena semua fitur sudah numerik/ordinal, one-hot encoding juga tidak diperlukan kecuali versi dataset lain menambahkan kolom kategorikal.

## 15. Simpan Model Terbaik untuk Deployment

Model terbaik dilatih ulang pada seluruh dataset, lalu disimpan sebagai file `.joblib`. Metadata fitur juga disimpan agar aplikasi Flask bisa membuat form input dan membaca label prediksi dengan konsisten.

In [ ]:
# Menyimpan model terbaik untuk kebutuhan deployment Flask.
import json
import joblib
from datetime import datetime
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "stress_level_model.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata.json"
METRICS_PATH = MODEL_DIR / "model_metrics.json"

# Latih ulang model terbaik pada seluruh data agar model deployment memakai data latih sebanyak mungkin.
deployment_model = model_pipelines[best_model_name]
deployment_model.fit(X, y)
joblib.dump(deployment_model, MODEL_PATH)

feature_schema = []
for column in X.columns:
    feature_schema.append({
        "name": column,
        "label": column.replace("_", " ").title(),
        "description": column.replace("_", " "),
        "min": int(X[column].min()),
        "max": int(X[column].max()),
        "step": 1,
        "default": int(round(float(X[column].median()))),
    })

metadata = {
    "model_name": best_model_name,
    "model_type": "sklearn.pipeline.Pipeline",
    "target": TARGET_COL,
    "features": list(X.columns),
    "stress_labels": {str(key): value for key, value in stress_labels.items()},
    "feature_schema": feature_schema,
    "dataset_path": str(DATA_PATH),
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "random_state": RANDOM_STATE,
}

metrics = {
    "test_accuracy": float(best_accuracy),
    "test_f1_macro": float(best_f1_macro),
    "selected_by": "F1 Macro Mean from 5-fold CV on train set",
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Model berhasil disimpan ke   : {MODEL_PATH}")
print(f"Metadata berhasil disimpan ke: {METADATA_PATH}")
print(f"Metrics berhasil disimpan ke : {METRICS_PATH}")

## 16. Menjalankan Web Flask

Aplikasi Flask tersedia di `app.py`. Jalankan perintah berikut dari root proyek, lalu buka alamat lokal yang muncul di terminal.

In [ ]:
# Jalankan di terminal, bukan di cell notebook, agar server Flask tetap aktif.
# python app.py